In [1]:
# 03. A/Bテスト分析

## 目的

新作商品・おすすめ表示などのデジタル施策を想定し、
Control群とTreatment群のユーザー行動を比較する。

本分析では、以下のKPIを比較する。

- クリック率（CTR）
- 購入率（CVR）
- 平均購入金額
- 売上
- 購入率の改善幅（Lift）

さらに、購入率についてカイ二乗検定を実施し、
Control群とTreatment群の差が統計的に有意かを検証する。

※本分析に使用するデータは、公開情報や一般的な書店・EC市場の構造を参考に
分析手法の実証を目的として設計した疑似データであり、
実在する企業の内部データではない。

SyntaxError: invalid character '、' (U+3001) (<ipython-input-1-49ba03aec4af>, line 5)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

In [3]:
df = pd.read_csv("dataset/ab_test.csv")

print("データ件数:", len(df))
print("カラム数:", len(df.columns))

display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/ab_test.csv'

In [4]:
df.info()

NameError: name 'df' is not defined

In [5]:
required_columns = [
    "experiment_id",
    "user_id",
    "experiment_group",
    "exposure_date",
    "exposed",
    "clicked",
    "purchased",
    "order_value"
]

# 必須カラムの存在確認
assert all(
    column in df.columns
    for column in required_columns
), "必須カラムが不足しています。"

# Control / Treatment の確認
assert set(df["experiment_group"].unique()) == {
    "Control",
    "Treatment"
}, "実験グループが想定と異なります。"

# ユーザーIDの重複確認
assert df["user_id"].is_unique, "user_id に重複があります。"

# フラグ値の確認
assert set(df["exposed"].unique()).issubset({0, 1})
assert set(df["clicked"].unique()).issubset({0, 1})
assert set(df["purchased"].unique()).issubset({0, 1})

# 購入していないユーザーの購入金額が0であることを確認
assert (
    df.loc[df["purchased"] == 0, "order_value"] == 0
).all(), "未購入ユーザーに購入金額があります。"

print("✓ データ品質チェック完了")

NameError: name 'df' is not defined

In [6]:
group_counts = (
    df.groupby("experiment_group")
      .size()
      .reset_index(name="users")
)

display(group_counts)

NameError: name 'df' is not defined

In [7]:
summary = (
    df.groupby("experiment_group")
      .agg(
          users=("user_id", "count"),
          clicks=("clicked", "sum"),
          purchasers=("purchased", "sum"),
          revenue=("order_value", "sum")
      )
      .reset_index()
)

summary["ctr"] = (
    summary["clicks"] / summary["users"]
)

summary["conversion_rate"] = (
    summary["purchasers"] / summary["users"]
)

summary["avg_order_value"] = np.where(
    summary["purchasers"] > 0,
    summary["revenue"] / summary["purchasers"],
    0
)

display(summary)

NameError: name 'df' is not defined

In [8]:
display(
    summary.style.format({
        "ctr": "{:.1%}",
        "conversion_rate": "{:.1%}",
        "revenue": "${:,.2f}",
        "avg_order_value": "${:,.2f}"
    })
)

NameError: name 'summary' is not defined

In [9]:
control_rate = summary.loc[
    summary["experiment_group"] == "Control",
    "conversion_rate"
].iloc[0]

treatment_rate = summary.loc[
    summary["experiment_group"] == "Treatment",
    "conversion_rate"
].iloc[0]

absolute_lift = treatment_rate - control_rate

relative_lift = (
    (treatment_rate / control_rate) - 1
) * 100

print(f"Control 購入率: {control_rate:.1%}")
print(f"Treatment 購入率: {treatment_rate:.1%}")
print(f"購入率の差: {absolute_lift:+.1%}")
print(f"相対改善率（Lift）: {relative_lift:+.1f}%")

NameError: name 'summary' is not defined

In [10]:
control_ctr = summary.loc[
    summary["experiment_group"] == "Control",
    "ctr"
].iloc[0]

treatment_ctr = summary.loc[
    summary["experiment_group"] == "Treatment",
    "ctr"
].iloc[0]

ctr_lift = (
    (treatment_ctr / control_ctr) - 1
) * 100

print(f"Control CTR: {control_ctr:.1%}")
print(f"Treatment CTR: {treatment_ctr:.1%}")
print(f"CTR相対改善率: {ctr_lift:+.1f}%")

NameError: name 'summary' is not defined

In [11]:
contingency_table = pd.crosstab(
    df["experiment_group"],
    df["purchased"]
)

print("購入有無のクロス集計")
display(contingency_table)

NameError: name 'df' is not defined

In [12]:
chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print(f"χ²統計量: {chi2:.4f}")
print(f"自由度: {dof}")
print(f"p値: {p_value:.4f}")

NameError: name 'contingency_table' is not defined

In [13]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

print("カイ二乗検定における期待度数")
display(expected_df)

NameError: name 'expected' is not defined

In [14]:
alpha = 0.05

if p_value < alpha:
    significance_result = "統計的に有意な差が確認されました。"
else:
    significance_result = "統計的に有意な差は確認できませんでした。"

print(significance_result)
print(f"有意水準: α = {alpha}")

NameError: name 'p_value' is not defined

In [15]:
if p_value < alpha:
    conclusion = (
        f"Treatment群の購入率は{treatment_rate:.1%}で、"
        f"Control群の{control_rate:.1%}を上回った。"
        f"カイ二乗検定の結果、p={p_value:.4f}となり、"
        f"有意水準5%において統計的に有意な差が確認された。"
    )
else:
    conclusion = (
        f"Treatment群の購入率は{treatment_rate:.1%}、"
        f"Control群は{control_rate:.1%}であった。"
        f"カイ二乗検定の結果、p={p_value:.4f}となり、"
        f"有意水準5%では統計的に有意な差は確認できなかった。"
    )

print(conclusion)

NameError: name 'p_value' is not defined

In [16]:
plot_df = summary.set_index("experiment_group")

plt.figure(figsize=(7, 5))

plt.bar(
    plot_df.index,
    plot_df["conversion_rate"]
)

plt.title("Conversion Rate by Experiment Group")
plt.ylabel("Conversion Rate")
plt.xlabel("Experiment Group")
plt.ylim(0, max(plot_df["conversion_rate"]) * 1.3)

for i, value in enumerate(plot_df["conversion_rate"]):
    plt.text(
        i,
        value,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'summary' is not defined

In [17]:
plt.figure(figsize=(7, 5))

plt.bar(
    plot_df.index,
    plot_df["ctr"]
)

plt.title("CTR by Experiment Group")
plt.ylabel("CTR")
plt.xlabel("Experiment Group")
plt.ylim(0, max(plot_df["ctr"]) * 1.3)

for i, value in enumerate(plot_df["ctr"]):
    plt.text(
        i,
        value,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'plot_df' is not defined

<Figure size 700x500 with 0 Axes>

In [18]:
plt.figure(figsize=(7, 5))

plt.bar(
    plot_df.index,
    plot_df["avg_order_value"]
)

plt.title("Average Order Value by Experiment Group")
plt.ylabel("Average Order Value")
plt.xlabel("Experiment Group")

for i, value in enumerate(plot_df["avg_order_value"]):
    plt.text(
        i,
        value,
        f"${value:.2f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'plot_df' is not defined

<Figure size 700x500 with 0 Axes>

In [19]:
result_summary = pd.DataFrame({
    "Metric": [
        "Control CVR",
        "Treatment CVR",
        "CVR Lift",
        "Control CTR",
        "Treatment CTR",
        "CTR Lift",
        "Chi-square p-value",
        "Statistically Significant"
    ],
    "Value": [
        f"{control_rate:.1%}",
        f"{treatment_rate:.1%}",
        f"{relative_lift:+.1f}%",
        f"{control_ctr:.1%}",
        f"{treatment_ctr:.1%}",
        f"{ctr_lift:+.1f}%",
        f"{p_value:.4f}",
        "Yes" if p_value < alpha else "No"
    ]
})

display(result_summary)

NameError: name 'control_rate' is not defined

In [20]:
## Business Insight

### Fact

Control群とTreatment群について、クリック率・購入率・平均購入金額を比較した。

また、購入率についてカイ二乗検定を実施し、
観測された差が統計的に有意かを検証した。

### Interpretation

Treatment群のKPIがControl群を上回った場合、
想定したデジタル施策がユーザーの購買行動に影響した可能性が考えられる。

ただし、本分析で使用しているデータは疑似データであるため、
実在する書店・EC事業における施策効果を直接示すものではない。

### Next Action

実務で同様の施策を実施する場合は、

1. より大規模なサンプルでA/Bテストを実施する
2. テスト期間を延長して再現性を確認する
3. 新規顧客・既存顧客などのセグメント別に効果を比較する
4. 購入率だけでなく平均購入金額やLTVも評価する
5. 統計的有意差だけでなく、ビジネス上の効果量も確認する

といった追加検証が必要となる。

SyntaxError: invalid character '、' (U+3001) (<ipython-input-20-bd42c05e732c>, line 5)